# GSoC next-year organization prediction

This notebook uses the five-year workbook history as a rolling supervised-learning problem. The training label is `T-1`; its features are `T-2` through `T-5`. After evaluation, the model is refit on all organizations and applied to `T-1` through `T-4` to predict participation in the next/current target year `T`.

The prediction is about organization presence in the GSoC list, not acceptance of a particular project.

In [20]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

candidates = [Path.cwd(), Path.cwd() / 'src' / 'backend']
BACKEND = next((path for path in candidates if (path / 'gsoc_organizations.xlsx').exists()), candidates[-1])
WORKBOOK = BACKEND / 'gsoc_organizations.xlsx'
PREDICTIONS = BACKEND / 'gsoc_t_year_predictions.csv'
assert WORKBOOK.exists(), f'Workbook not found: {WORKBOOK}'


In [21]:
orgs = pd.read_excel(WORKBOOK, sheet_name='Orgs')
scrape_log = pd.read_excel(WORKBOOK, sheet_name='Scrape log')
required = ['Organization', 'T-1', 'T-2', 'T-3', 'T-4', 'T-5']
missing = sorted(set(required) - set(orgs.columns))
if missing:
    raise ValueError(f'Missing required columns: {missing}')

scrape_log['year'] = pd.to_numeric(scrape_log['year'], errors='coerce')
scrape_log['total_orgs'] = pd.to_numeric(scrape_log['total_orgs'], errors='coerce')
scrape_log = scrape_log.dropna(subset=['year']).astype({'year': int})
current_year = int(scrape_log['year'].max())
current_total = int(scrape_log.loc[scrape_log['year'].eq(current_year), 'total_orgs'].iloc[0])
target_year = current_year + 1 if current_total > 0 else current_year

presence_cols = ['T-1', 'T-2', 'T-3', 'T-4', 'T-5']
for col in presence_cols:
    orgs[col] = orgs[col].map({True: 1, False: 0, 1: 1, 0: 0}).astype('Int64')

print(f'Organizations: {len(orgs):,}')
print(f'Current scraped year: {current_year}; prediction target T: {target_year}')
print(scrape_log.to_string(index=False))


Organizations: 308
Current scraped year: 2026; prediction target T: 2027
 year  total_orgs
 2026         183
 2025         185
 2024         194
 2023         168
 2022         198


## Shifted training setup

For every organization, the training target is `T-1`. The four input features are the four older flags. This makes the model learn the transition from an organization's recent participation history to the next year. The same four-position feature layout is then shifted forward for the T-year prediction.

In [22]:
feature_cols = ['T-2', 'T-3', 'T-4', 'T-5']
X = orgs[feature_cols].fillna(0).astype(float)
y = orgs['T-1'].fillna(0).astype(int)

models = {
    'logistic_regression': make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42))
}
if y.nunique() < 2:
    raise ValueError('T-1 has only one class; a classifier cannot be trained.')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []
for name, model in models.items():
    result = cross_validate(model, X, y, cv=cv, scoring=['accuracy', 'balanced_accuracy', 'roc_auc'])
    scores.append({
        'model': name,
        'accuracy_mean': result['test_accuracy'].mean(),
        'balanced_accuracy_mean': result['test_balanced_accuracy'].mean(),
        'roc_auc_mean': result['test_roc_auc'].mean(),
    })
scores = pd.DataFrame(scores).sort_values(['balanced_accuracy_mean', 'roc_auc_mean'], ascending=False)
display(scores)


,model,accuracy_mean,balanced_accuracy_mean,roc_auc_mean
0,logistic_regression,0.740455,0.730616,0.764739


In [23]:
best_name = scores.iloc[0]['model']
best_model = clone(models[best_name]).fit(X, y)

# Shift the feature window forward one year: T-1 becomes the newest input.
prediction_features = ['T-1', 'T-2', 'T-3', 'T-4']
X_target = orgs[prediction_features].fillna(0).astype(float)
# Keep the feature names identical to training; only the values move forward one year.
X_target.columns = feature_cols
predicted = best_model.predict(X_target).astype(bool)
probability = best_model.predict_proba(X_target)[:, 1]

# Keep every scraped workbook column alongside the new prediction columns.
predictions = orgs.copy()
predictions[f'Predicted {target_year}'] = predicted
predictions[f'Probability {target_year}'] = probability.round(4)
predictions['Model'] = best_name
predictions['Training target'] = 'T-1'
predictions.to_csv(PREDICTIONS, index=False)

print(f'Best model: {best_name}')
print(f'Predicted organizations for T={target_year}: {int(predicted.sum()):,} of {len(predicted):,}')
display(predictions.sort_values(f'Probability {target_year}', ascending=False).head(20))
print(f'Saved predictions to {PREDICTIONS}')


Best model: logistic_regression
Predicted organizations for T=2027: 183 of 308


,Organization,Description,Long Description,Tech,Topic,Website,GSoC-link,Contributor-guidance,T-1,T-2,T-3,T-4,T-5,Years,Predicted 2027,Probability 2027,Model,Training target
145,Kubeflow,The Machine Learning Toolkit for Kubernetes (AI),Our goal is to streamline the process of scali...,"python, go, kubernetes, typescript, YAML","machine learning, kubernetes, AI/ML, generativ...",https://kubeflow.org,https://summerofcode.withgoogle.com/programs/2...,https://www.kubeflow.org/events/upcoming-event...,1,1,1,0,0,"2026, 2025, 2024",True,0.7792,logistic_regression,T-1
159,Open Science Initiative for Perfusion Imaging,Open access resources for perfusion imaging,Perfusion Magnetic Resonance Imaging (MRI) is ...,"python, github","artificial intelligence, data visualization, d...",https://osipi.ismrm.org/,https://summerofcode.withgoogle.com/programs/2...,https://docs.google.com/document/d/e/2PACX-1vT...,1,1,1,0,0,"2026, 2025, 2024",True,0.7792,logistic_regression,T-1
130,Data for the Common Good,Connect. Share. Discover.,Data for the Common Good is dedicated to build...,"python, javascript, kubernetes, reactjs, terra...","web, health, data, infrastructure, software",https://commons.cri.uchicago.edu,https://summerofcode.withgoogle.com/programs/2...,https://chicagopcdc.github.io/GSoC/ideas,1,1,1,0,0,"2026, 2025, 2024",True,0.7792,logistic_regression,T-1
146,LabLua,Programming Language Research with emphasis on...,LabLua is a research lab at the Catholic Unive...,"lua, luarocks, kernel, lunatik, pallene","compilers, scripting languages, kernel scripti...",http://www.lua.inf.puc-rio.br,https://summerofcode.withgoogle.com/programs/2...,https://github.com/labluapucrio/gsoc,1,1,1,0,0,"2026, 2025, 2024",True,0.7792,logistic_regression,T-1
158,Open HealthCare Network,Reimagining Healthcare Delivery,"Open Healthcare Network (OHC), originally esta...","python, django, react, typescript, NextJs","electronic medical records, Digital Public Goo...",https://ohc.network/,https://summerofcode.withgoogle.com/programs/2...,https://github.com/ohcnetwork/care_fe/issues?q...,1,1,1,0,0,"2026, 2025, 2024",True,0.7792,logistic_regression,T-1
143,JSON Schema,We enable the reliable use of JSON data format.,JSON Schema is a vocabulary that allows you to...,"python, javascript, typescript, .net, JSON Schema","web, apis, standards, data validation, develop...",https://json-schema.org/,https://summerofcode.withgoogle.com/programs/2...,https://github.com/json-schema-org/community/b...,1,1,1,0,0,"2026, 2025, 2024",True,0.7792,logistic_regression,T-1
160,Open Transit Software Foundation,Help make public transit better,We are the home for the OneBusAway project. On...,"android, java, golang, docker, ios","apps, Transit, travel, bus",https://opentransitsoftwarefoundation.org,https://summerofcode.withgoogle.com/programs/2...,https://opentransitsoftwarefoundation.org/2026...,1,1,1,0,0,"2026, 2025, 2024",True,0.7792,logistic_regression,T-1
162,OpenELIS Global,Empowering labs to Ensure Quality Health Care,OpenELIS Global (Open Enterprise laboratory In...,"postgresql, javascript, java, react, spring","Health Care, Laboratory Information System",https://openelis-global.org/,https://summerofcode.withgoogle.com/programs/2...,https://uwdigi.atlassian.net/wiki/spaces/OG/pa...,1,1,1,0,0,"2026, 2025, 2024",True,0.7792,logistic_regression,T-1
139,Graphite,Multimedia art/design editor and graphics engine,Graphite is an in-development raster and vecto...,"rust, vulkan, webgpu","compilers, programming languages, graphics, co...",https://graphite.art,https://summerofcode.withgoogle.com/programs/2...,https://graphite.art/volunteer/guide/student-p...,1,1,1,0,0,"2026, 2025, 2024",True,0.7792,logistic_regression,T-1
127,API Dash,Next-gen AI powered Open Source API DevTool,API Dash is a beautiful open-source cross-plat...,"python, react, flutter, typescript, ai","api, developer tools, ai, Agents, MCP",https://apidash.dev,https://summerofcode.withgoogle.com/programs/2...,https://github.com/foss42/

Saved predictions to c:\Users\hridd\Documents\VSCode\Web_Dev\Projects\Advanced\orglytics\src\backend\gsoc_t_year_predictions.csv


In [24]:
predictions

,Organization,Description,Long Description,Tech,Topic,Website,GSoC-link,Contributor-guidance,T-1,T-2,T-3,T-4,T-5,Years,Predicted 2027,Probability 2027,Model,Training target
0,52°North Spatial Information Research GmbH,Innovative ideas & technologies in geoinformatics,52°North is an open source initiative in the f...,"javascript, android, java, web services, ogc s...","citizen science, spatial information infrastru...",https://52north.org/,https://summerofcode.withgoogle.com/programs/2...,https://52north.org/outreach-dissemination/goo...,1,1,1,1,1,"2026, 2025, 2024, 2023, 2022",True,0.7277,logistic_regression,T-1
1,AboutCode,"Scan code for origin, license and vulnerabilities",AboutCode.org is a community of open source de...,"python, javascript, Django+PostgreSQL, C/Rust/Go","dependencies, vulnerabilities, SoftwareComposi...",https://aboutcode.org,https://summerofcode.withgoogle.com/programs/2...,https://github.com/aboutcode-org/aboutcode/wik...,1,1,1,1,1,"2026, 2025, 2024, 2023, 2022",True,0.7277,logistic_regression,T-1
2,AOSSIE,Australian Umbrella Org for Open-Source Projects,"We are an Australian not-for-profit charity, f...","python, javascript, flutter, Blockchain, Solidity","web, artificial intelligence, communication, m...",https://www.aossie.org,https://summerofcode.withgoogle.com/programs/2...,https://github.com/AOSSIE-Org/Info/blob/main/G...,1,1,1,1,1,"2026, 2025, 2024, 2023, 2022",True,0.7277,logistic_regression,T-1
3,Apache Software Foundation,Open source software to the public free of charge,The Foundation provides an established framewo...,"c, java, c++","big data, cloud, libraries, other",https://apache.org,https://summerofcode.withgoogle.com/programs/2...,https://s.apache.org/gsoc2026ideas,1,1,1,1,1,"2026, 2025, 2024, 2023, 2022",True,0.7277,logistic_regression,T-1
4,ArduPilot,World's most advanced autonomous vehicle software,ArduPilot is the world's most widely used open...,"python, lua, c++, pixhawk","robotics, ai, Drone, autonomous vehicle, unman...",https://ardupilot.org/,https://summerofcode.withgoogle.com/programs/2...,https://ardupilot.org/dev/docs/gsoc-ideas-list...,1,1,1,1,1,"2026, 2025, 2024, 2023, 2022",True,0.7277,logistic_regression,T-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
303,TianoCore,"Modern, open source, UEFI BIOS firmware","Welcome to TianoCore, the community supporting...","c, python, rust, golang, uefi","firmware, uefi, edk2, acpi, BIOS",https://www.tianocore.org/,https://summerofcode.withgoogle.com/programs/2...,https://github.com/tianocore/tianocore.github....,0,0,0,0,1,2022,False,0.2136,logistic_regression,T-1
304,Tiled,A flexible level editor for everyone!,Tiled is a 2D engine-agnostic level editor for...,"c++, qt","games, Level Editor",https://www.mapeditor.org,https://summerofcode.withgoogle.com/programs/2...,https://github.com/mapeditor/tiled/wiki/GSoC-2...,1,0,0,0,0,2026,True,0.5906,logistic_regression,T-1
305,United Nations Office of Information Communica...,Sustainable future through innovative technology.,"The Open Source Team at the United Nations, ba...","python, javascript, css","Technology, innovation",https://unite.un.org/en,https://summerofcode.withgoogle.com/programs/2...,https://opensource.unicc.org/un/unoict/mentors...,1,0,0,0,0,2026,True,0.5906,logistic_regression,T-1
306,Weaviate,The ML-first vector search engine,SeMI Technologies creates database software li...,"python, machine learning, golang, vectors, nat...","machine learning, Vector Database, Search Engines",https://weaviate.io,https://summerofcode.withgoogle.com/programs/2...,https://weaviate.io/google-summer/gsoc-23,0,0,0,0,1,2022,False,0.2136,logistic_regression,T-1
